In [1]:
from pathlib import Path
import pyodbc
import toml
import pandas as pd

config_path = Path.cwd().parents[0] / "config" / "config.toml"
cfg = toml.load(config_path)

db = cfg["database"]

conn_str = (
    f"DRIVER={{{db['driver']}}};"
    f"SERVER={db['server']};"
    f"DATABASE={db['database']};"
    f"Trusted_Connection={db['trusted_connection']};"
    f"TrustServerCertificate=yes;"
)

conn = pyodbc.connect(conn_str)

In [4]:
import pandas as pd
from datetime import date

query = """
WITH base_kpi AS (
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN Revenue IS NULL THEN 1 ELSE 0 END) AS null_revenue_rows,
        COUNT(DISTINCT SalesOrderID) AS distinct_orders,
        SUM(Revenue) AS total_revenue,
        COUNT(DISTINCT CustomerID) AS customer_count,
        SUM(Revenue) * 1.0 / NULLIF(COUNT(DISTINCT SalesOrderID), 0) AS AOV,
        MAX(CAST(OrderDate AS date)) AS latest_order_date
    FROM dbo.vw_fact_sales
),

daily_sales AS (
    SELECT
        CAST(OrderDate AS date) AS order_date,
        SUM(Revenue) AS daily_revenue
    FROM dbo.vw_fact_sales
    GROUP BY CAST(OrderDate AS date)
),

latest_revenue_check AS (
    SELECT TOP 1
        order_date,
        daily_revenue,
        AVG(daily_revenue) OVER (
            ORDER BY order_date
            ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING
        ) AS prev_7day_avg
    FROM daily_sales
    ORDER BY order_date DESC
),

category_check AS (
    SELECT
        COUNT(*) AS zero_revenue_category_count
    FROM (
        SELECT
            ProductCategory
        FROM dbo.vw_fact_sales
        GROUP BY ProductCategory
        HAVING SUM(Revenue) = 0
    ) z
)

SELECT
    b.total_rows,
    b.null_revenue_rows,
    b.distinct_orders,
    b.total_revenue,
    b.customer_count,
    b.AOV,
    b.latest_order_date,

    r.order_date AS latest_revenue_date,
    r.daily_revenue,
    r.prev_7day_avg,

    CASE
        WHEN r.prev_7day_avg IS NULL THEN NULL
        ELSE r.daily_revenue * 1.0 / NULLIF(r.prev_7day_avg, 0)
    END AS revenue_ratio,

    CASE
        WHEN r.prev_7day_avg IS NULL THEN 0
        WHEN r.daily_revenue < r.prev_7day_avg * 0.5 THEN 1
        ELSE 0
    END AS revenue_drop_flag,

    c.zero_revenue_category_count,

    CASE
        WHEN c.zero_revenue_category_count > 0 THEN 1
        ELSE 0
    END AS zero_revenue_category_flag

FROM base_kpi b
CROSS JOIN latest_revenue_check r
CROSS JOIN category_check c;
"""

validation_df = pd.read_sql(query, conn)
df = validation_df

if df.loc[0, "total_rows"] == 0:
    raise Exception("❌ No data in fact table")

if df.loc[0, "null_revenue_rows"] > 0:
    raise Exception("❌ Revenue contains NULLs")

if pd.notna(df.loc[0, "revenue_drop_flag"]) and df.loc[0, "revenue_drop_flag"] == 1:
    raise Exception("❌ Revenue dropped significantly vs previous 7-day average")

if df.loc[0, "zero_revenue_category_flag"] == 1:
    raise Exception("❌ One or more product categories have zero revenue")

# latest_date = pd.to_datetime(df.loc[0, "latest_order_date"]).date()

# if (date.today() - latest_date).days > 2:
    raise Exception(f"❌ Data is stale. Latest order date is {latest_date}")

print("✅ Validation passed")
print(df)

C:\Users\hisuk\AppData\Local\Temp\ipykernel_24112\1849907515.py:85: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  validation_df = pd.read_sql(query, conn)


✅ Validation passed
   total_rows  null_revenue_rows  distinct_orders  total_revenue  \
0      121317                  0            31465   1.098464e+08   

   customer_count          AOV latest_order_date latest_revenue_date  \
0           19119  3491.065672        2025-06-29          2025-06-29   

   daily_revenue  prev_7day_avg  revenue_ratio  revenue_drop_flag  \
0        2643.61        1668.97       1.583976                  0   

   zero_revenue_category_count  zero_revenue_category_flag  
0                            0                           0  
